# 🚀 Image Captioning — GPU Model Trainer (Google Colab)

> **Before running:** Go to `Runtime → Change runtime type → T4 GPU`

This notebook will:
1. Verify GPU availability
2. Clone your project from GitHub
3. Install dependencies
4. Mount Google Drive to persist trained model
5. Run the full training pipeline with EarlyStopping & checkpoints
6. Plot training history

## ✅ Step 1 — Verify GPU

In [1]:
!nvidia-smi

Tue Jul  7 12:42:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   57C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print(f'Num GPUs Available: {len(gpus)}')
for gpu in gpus:
    print(f'  → {gpu}')

if not gpus:
    raise RuntimeError('❌ No GPU detected! Go to Runtime → Change runtime type → GPU')

for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

print('\n✅ GPU ready!')

Num GPUs Available: 1
  → PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')

✅ GPU ready!


## 📁 Step 2 — Mount Google Drive

In [3]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_SAVE_DIR = '/content/drive/MyDrive/imcaption_trained'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print(f'📂 Model will be saved to: {DRIVE_SAVE_DIR}')

Mounted at /content/drive
📂 Model will be saved to: /content/drive/MyDrive/imcaption_trained


## 📦 Step 3 — Clone Repo & Install Dependencies

> 💡 Replace the `REPO_URL` with your actual GitHub repository URL.

In [4]:
import os

REPO_URL = 'https://github.com/divyanshrajput118/imcaption.git' 

if not os.path.exists('/content/imcaption'):
    os.system(f'git clone {REPO_URL} /content/imcaption')
else:
    print('Repo already cloned. Pulling latest...')
    os.system('cd /content/imcaption && git pull')

os.chdir('/content/imcaption')
print(f'Working directory: {os.getcwd()}')

Working directory: /content/imcaption


In [10]:
!git pull

remote: Enumerating objects: 20, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 12 (delta 7), reused 12 (delta 7), pack-reused 0 (from 0)
Unpacking objects: 100% (12/12), 11.93 KiB | 718.00 KiB/s, done.
From https://github.com/divyanshrajput118/imcaption
   b8a6b7b..b8668bd  main       -> origin/main
Updating b8a6b7b..b8668bd
Fast-forward
 research/04_model_trainer.ipynb                  | 112 ++----------
 research/colab_model_trainer.ipynb               | 208 +++++++++++++++--------
 src/imgCaption/components/data_transformation.py |   4 +-
 src/imgCaption/components/model_trainer.py       |  94 ++++++----
 4 files changed, 216 insertions(+), 202 deletions(-)


In [5]:
!pip install -e . -q
!pip install -r requirements_colab.txt -q
print('✅ Dependencies installed!')

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 107.4 MB/s eta 0:00:0000:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.8/469.8 kB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 120.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 82.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 131.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.3/79.3 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 451.2/451.2 kB 37.5 MB/s eta

## 📥 Step 4 — Check / Load Artifacts

> ⚡ If you have artifacts saved in Drive from a previous run, set `COPY_FROM_DRIVE = True`.

In [11]:
!python main.py

[2026-07-07 13:19:14,605: INFO: utils: NumExpr defaulting to 2 threads.]
[2026-07-07 13:19:20,079: INFO: main: >>>>>> stage Data_Ingestion_Stage started <<<<<<]
[2026-07-07 13:19:20,083: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-07-07 13:19:20,085: INFO: common: yaml file: params.yaml loaded successfully]
[2026-07-07 13:19:20,085: INFO: common: created directory at: artifacts]
[2026-07-07 13:19:20,086: INFO: common: created directory at: artifacts/data_ingestion]
[2026-07-07 13:19:20,086: INFO: data_ingestion: Downloading data from https://drive.google.com/file/d/1X-l4QnuhIYMXm-Lgl2N0MO-xR75ly3OD/view?usp=sharing into file artifacts/data_ingestion/data.zip]
Downloading...
From (original): https://drive.google.com/uc?id=1X-l4QnuhIYMXm-Lgl2N0MO-xR75ly3OD
From (redirected): https://drive.google.com/uc?id=1X-l4QnuhIYMXm-Lgl2N0MO-xR75ly3OD&confirm=t&uuid=9afda380-1156-4f56-8df2-8093e9840ef9
To: /content/imcaption/artifacts/data_ingestion/data.zip
100% 1.11G/1.11

In [14]:
# Run this immediately after !python main.py completes
import shutil, os

# Save the trained model to Drive
shutil.copy(
    '/content/imcaption/artifacts/training/model.keras',
    '/content/drive/MyDrive/imcaption_trained/modelV1.keras'
)
print("✅ Trained model saved to Drive!")


✅ Trained model saved to Drive!


In [15]:
import os

required = [
    'artifacts/data_ingestion/Images',
    'artifacts/data_transformation/vectorizer_data.pkl',
    'artifacts/data_transformation/train_images_captions.pkl',
    'artifacts/prepare_base_model/feature_extractor.keras',
    'artifacts/prepare_base_model/main_model.keras',
]

missing = [p for p in required if not os.path.exists(p)]

if missing:
    print('⚠️  Missing artifacts:')
    for m in missing:
        print(f'  ✗ {m}')
    print('\nEither set COPY_FROM_DRIVE=True below or run the full pipeline.')
else:
    print('✅ All required artifacts found!')

✅ All required artifacts found!
